# Colab Dense Embedding & Indexing - LegalRAG

Notebook này hỗ trợ xây dựng **Dense Vector Index** cho **toàn bộ các chiến lược chunking** và **2 mô hình embedding**:
- **MiniLM-L12-v2** (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` - 384 chiều)
- **BGE-M3** (`BAAI/bge-m3` - 1024 chiều)

⚡ **Đã chia thành các cell độc lập kèm cơ chế dọn VRAM tự động (`empty_cache`)** để tránh lỗi tràn bộ nhớ (Out of Memory).
Bạn có thể chạy lần lượt từng cell bên dưới.

In [ ]:
# 1. Kiểm tra GPU và môi trường PyTorch CUDA
!nvidia-smi
import torch

print("\n" + "=" * 50)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM:      {vram_gb:.2f} GB")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> T4 GPU để tăng tốc.")
print("=" * 50)

In [ ]:
# 2. Clone mã nguồn LegalRAG và chuẩn bị môi trường
import os
import sys
from pathlib import Path

target_dir = Path("/content/LegalRAG")

if not (target_dir / "src" / "legal_rag").exists():
    print("Mã nguồn chưa tồn tại. Đang clone mới...")
    !rm -rf /content/LegalRAG
    !git clone https://github.com/ngminhduc129/LegalRAG.git /content/LegalRAG
else:
    print("Cập nhật mã nguồn mới nhất...")
    !git -C /content/LegalRAG pull

%cd /content/LegalRAG

!pip install sentence-transformers underthesea rank-bm25 typer rich loguru pydantic pydantic-yaml --quiet

src_dir = str((target_dir / "src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import legal_rag
print(f"\n✓ Đã kết nối mã nguồn LegalRAG v{legal_rag.__version__} thành công từ: {src_dir}")

In [ ]:
# 3. Khởi tạo hàm Build Index & Tự động giải phóng VRAM (Chống OOM)
import gc
import torch
from pathlib import Path
from legal_rag.retrieval.dense import build_dense_index

output_base_dir = Path("/content/LegalRAG/data/indexes/dense")
output_base_dir.mkdir(parents=True, exist_ok=True)

# Thông tin 2 model
MODELS_CONFIG = {
    "minilm": {
        "name": "MiniLM-L12-v2",
        "model_id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "batch_size": 32,
    },
    "bge_m3": {
        "name": "BGE-M3",
        "model_id": "BAAI/bge-m3",
        "batch_size": 16,  # Giữ batch_size vừa phải để tránh tràn VRAM
    },
}

def clear_gpu_memory():
    """Thu hồi bộ nhớ Python và xả cache VRAM GPU."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def run_indexing(strat_name: str, chunk_file: Path, model_keys=["minilm", "bge_m3"], skip_existing=False):
    """Hàm build index chuyên biệt, giải phóng VRAM ngay sau khi lưu file."""
    if not chunk_file.exists():
        print(f"❌ Không tìm thấy file chunk: {chunk_file}")
        return
        
    for m_key in model_keys:
        m_info = MODELS_CONFIG[m_key]
        out_path = output_base_dir / f"dense_{strat_name}_{m_key}.pkl"
        
        if skip_existing and out_path.exists():
            print(f"⏩ [Bỏ qua] Index đã tồn tại: {out_path.name}")
            continue
            
        print(f"\n--- [BUILD] '{strat_name}' với {m_info['name']} ---")
        print(f"  Input : {chunk_file}")
        print(f"  Output: {out_path.name}")
        
        # Dọn dẹp VRAM trước khi load model
        clear_gpu_memory()
        
        idx = build_dense_index(
            chunks_path=chunk_file,
            index_path=out_path,
            model_name=m_info["model_id"],
            batch_size=m_info["batch_size"],
            max_length=256,
        )
        
        print(f"✓ Hoàn thành: {len(idx.documents)} docs | Shape: {idx.embeddings.shape}")
        
        # QUAN TRỌNG: Xóa đối tượng index và xả sạch VRAM ngay sau khi lưu file
        del idx
        clear_gpu_memory()

print("✓ Đã sẵn sàng hàm build_dense_index kèm chống OOM!")

### 3.1 Nhóm đề xuất: `legal_aware`

In [ ]:
# 3.1 Build Index cho phương pháp đề xuất: legal_aware (MiniLM + BGE-M3)
run_indexing(
    strat_name="legal_aware",
    chunk_file=Path("data/chunks/legal_aware_run06/labor_code_2019_legal_aware.jsonl"),
    model_keys=["minilm", "bge_m3"],
    skip_existing=False
)

### 3.2 Nhóm Baseline cấu trúc pháp lý: `clause` & `article`

In [ ]:
# 3.2 Build Index cho clause & article
# 1. Clause
run_indexing(
    strat_name="clause",
    chunk_file=Path("data/chunks/clause_run06/labor_code_2019_clause.jsonl"),
    model_keys=["minilm", "bge_m3"],
    skip_existing=False
)

# 2. Article
run_indexing(
    strat_name="article",
    chunk_file=Path("data/chunks/article_run06/labor_code_2019_article.jsonl"),
    model_keys=["minilm", "bge_m3"],
    skip_existing=False
)

### 3.3 Nhóm Baseline kích thước cố định: `fixed_300`, `fixed_500`, `fixed_800`

In [ ]:
# 3.3 Build Index cho nhóm Fixed Size (300, 500, 800 words)
fixed_sizes = ["fixed_300", "fixed_500", "fixed_800"]

for fs in fixed_sizes:
    chunk_path = Path(f"data/chunks/{fs}_run06/labor_code_2019_{fs}.jsonl")
    run_indexing(
        strat_name=fs,
        chunk_file=chunk_path,
        model_keys=["minilm", "bge_m3"],
        skip_existing=False
    )

### 3.4 Nhóm Baseline đoạn văn: `paragraph`

In [ ]:
# 3.4 Build Index cho paragraph
run_indexing(
    strat_name="paragraph",
    chunk_file=Path("data/chunks/paragraph_run06/labor_code_2019_paragraph.jsonl"),
    model_keys=["minilm", "bge_m3"],
    skip_existing=False
)

### 4. Kiểm tra truy vấn & Semantic Search Test

In [ ]:
# 4. Thử nghiệm truy vấn trên các Index đã build thành công
from legal_rag.retrieval.dense import DenseIndex

test_query = "Người lao động có quyền đơn phương chấm dứt hợp đồng lao động khi nào?"
index_files = sorted(list(output_base_dir.glob("*.pkl")))

print("=" * 80)
print(f"TÌM KIẾM THỬ NGHIỆM TRÊN CÁC INDEX HOÀN THÀNH:\n\"{test_query}\"")
print("=" * 80)

for pkl_file in index_files:
    print(f"\n>>> Index: [{pkl_file.name}] <<<")
    idx = DenseIndex.load(pkl_file)
    results = idx.search(test_query, k=2)
    for rank, (doc_id, score, doc) in enumerate(results, 1):
        art = doc.get("article", "N/A")
        title = doc.get("article_title", "")
        clause = doc.get("clause", "")
        prefix = f"Điều {art}"
        if clause:
            prefix += f", Khoản {clause}"
        print(f"  Top {rank} (Cosine: {score:.4f}): {prefix} - {title}")
        snippet = doc.get("content", "").replace("\n", " ").strip()
        print(f"    Nội dung: {snippet[:110]}...")
        
    del idx
    clear_gpu_memory()

### 5. Tổng kết & Tải toàn bộ Index (ZIP)

In [ ]:
# 5. Liệt kê dung lượng và nén file ZIP để tải về máy
import shutil

index_files = sorted(list(output_base_dir.glob("*.pkl")))
print(f"Đã tạo thành công {len(index_files)} file index:\n")
for f in index_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"- {f.name:35} : {size_mb:.2f} MB")

zip_output = Path("/content/LegalRAG_Dense_Indexes")
shutil.make_archive(str(zip_output), 'zip', str(output_base_dir))
zip_file = Path(f"{zip_output}.zip")
print(f"\n✓ Đã nén toàn bộ thành file: {zip_file.name} ({zip_file.stat().st_size / (1024*1024):.2f} MB)")

print("\nĐể tải file zip về máy, bỏ comment dòng dưới:")
# from google.colab import files; files.download(str(zip_file))